<a href="https://colab.research.google.com/github/austinmallie/ADS599_Capstone/blob/main/HospitalGeneral_FE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries / Setup

In [1]:
from google.colab import userdata
import os

token = userdata.get('Github')
owner = "austinmallie"
repo = "ADS599_Capstone"
repo_url = f"https://{token}@github.com/{owner}/{repo}.git"

# Only clone if the folder doesn't exist already
if not os.path.exists(repo):
    !git clone {repo_url}
else:
    print("Repo already cloned. Pulling latest changes...")
    %cd {repo}
    !git pull

%cd /content/{repo}

Cloning into 'ADS599_Capstone'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 167 (delta 45), reused 6 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 23.59 MiB | 7.72 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/ADS599_Capstone


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
#import data
df = pd.read_csv("Data-Folder/Hospital-Geneneral-Data/Hospital_General_Clean.csv")
df.head()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Count of Facility MORT Measures,Count of MORT Measures Better,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,Count of Facility Pt Exp Measures,Count of Facility TE Measures,Hospital_Type,Birthing_Center,Emergency_Services,Hospital_Ownership,Hospital_Rating
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,7.0,0.0,...,0.0,11.0,0.0,8.0,11.0,Acute Care,1,1,Government,4.0
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,6.0,0.0,...,0.0,8.0,1.0,8.0,12.0,Acute Care,1,1,Government,3.0
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,7.0,0.0,...,0.0,8.0,1.0,8.0,10.0,Acute Care,1,1,For-Profit,2.0
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,3.0,0.0,...,0.0,7.0,0.0,8.0,7.0,Acute Care,0,1,Non-Profit,1.0
4,010008,CRENSHAW COMMUNITY HOSPITAL,101 HOSPITAL CIRCLE,LUVERNE,AL,36049,CRENSHAW,(334) 335-3374,1.0,0.0,...,0.0,2.0,0.0,NaN,6.0,Acute Care,0,1,For-Profit,NaN


In [4]:
df.dtypes

,0
Facility ID,object
Facility Name,object
Address,object
City/Town,object
State,object
ZIP Code,int64
County/Parish,object
Telephone Number,object
Count of Facility MORT Measures,float64
Count of MORT Measures Better,float64


# Feature 1: Engagement Index


**Hypothesis**: Hospitals that report more measures within each domain signal a stronger administrative infrastructure



*   Engagement Index : tells you the average participation across all measures
*   Domains Reporting: Adds up across the domains how many of them have reporting



In [8]:
#max measures
MAX_MEASURES = {
    'MORT':   7,
    'Safety': 8,
    'READM':  11,
    'PtExp':  8,
    'TE':     12,
}

df['MORT_participation']   = df['Count of Facility MORT Measures'].fillna(0)   / MAX_MEASURES['MORT']
df['Safety_participation'] = df['Count of Facility Safety Measures'].fillna(0) / MAX_MEASURES['Safety']
df['READM_participation']  = df['Count of Facility READM Measures'].fillna(0)  / MAX_MEASURES['READM']
df['PtExp_participation']  = df['Count of Facility Pt Exp Measures'].fillna(0) / MAX_MEASURES['PtExp']
df['TE_participation']     = df['Count of Facility TE Measures'].fillna(0)     / MAX_MEASURES['TE']

participation_cols = [
    'MORT_participation', 'Safety_participation', 'READM_participation',
    'PtExp_participation', 'TE_participation'
]

# Overall engagement index (mean of all domain participations)
df['Engagement_Index'] = df[participation_cols].mean(axis=1)

# Count of domains with ANY reporting (0–5) — captures breadth
df['Domains_Reporting'] = (df[participation_cols] > 0).sum(axis=1)
df.head(3)

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Count of Facility MORT Measures,Count of MORT Measures Better,...,Emergency_Services,Hospital_Ownership,Hospital_Rating,MORT_participation,Safety_participation,READM_participation,PtExp_participation,TE_participation,Engagement_Index,Domains_Reporting
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,7.0,0.0,...,1,Government,4.0,1.000000,0.875,1.000000,1.0,0.916667,0.958333,5
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,6.0,0.0,...,1,Government,3.0,0.857143,0.875,0.818182,1.0,1.000000,0.910065,5
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,7.0,0.0,...,1,For-Profit,2.0,1.000000,1.000,0.818182,1.0,0.833333,0.930303,5


# Feature 2: Performance Signal Features

Original dataset shows us where the given facility performs stronger, worse or on average to other facilities. Using the total number of measures that were reported on, created a ratio for of that how many were either better or worse than the industry.

In [18]:
RATED_DOMAINS = {
'MORT':   {'better':   'Count of MORT Measures Better',
           'worse':    'Count of MORT Measures Worse',
           'reported': 'Count of Facility MORT Measures'},
'Safety': {'better':   'Count of Safety Measures Better',
           'worse':    'Count of Safety Measures Worse',
           'reported': 'Count of Facility Safety Measures'},
'READM':  {'better':   'Count of READM Measures Better',
           'worse':    'Count of READM Measures Worse',
           'reported': 'Count of Facility READM Measures'},
}

for domain, cols in RATED_DOMAINS.items():
  df[f'{domain}_better'] = df[cols['better']] / df[cols['reported']]
  df[f'{domain}_Worse']  = df[cols['worse']]  / df[cols['reported']]

  _total_better   = sum(df[cols['better']].fillna(0)   for cols in RATED_DOMAINS.values())
  _total_worse    = sum(df[cols['worse']].fillna(0)    for cols in RATED_DOMAINS.values())
  _total_reported = sum(df[cols['reported']].fillna(0) for cols in RATED_DOMAINS.values())

  df['Net_Performance_Score'] = _total_better - _total_worse

  df['Net_Performance_Rate'] = np.where(
      _total_reported > 0,
      (_total_better - _total_worse) / _total_reported,
      np.nan
  )

  df['Performance_Ratio'] = np.where(
      _total_reported > 0,
      _total_better / _total_reported,
      np.nan
  )

# Clean Dataset
Drop Redundant or unnecessary Columns

In [20]:
#get our columns
df.dtypes

,0
Facility ID,object
Facility Name,object
Address,object
City/Town,object
State,object
ZIP Code,int64
County/Parish,object
Telephone Number,object
Count of Facility MORT Measures,float64
Count of MORT Measures Better,float64


In [21]:
cols_to_drop = [
    # Hospital Details
    'Facility Name',
    'Address',
    'City/Town',
    'ZIP Code',
    'County/Parish',
    'Telephone Number',
    # Raw source measure counts
    'Count of Facility MORT Measures',
    'Count of Facility Safety Measures',
    'Count of Facility READM Measures',
    'Count of Facility Pt Exp Measures',
    'Count of Facility TE Measures',
    # Raw Better / No Different / Worse counts
    'Count of MORT Measures Better',
    'Count of MORT Measures No Different',
    'Count of MORT Measures Worse',
    'Count of Safety Measures Better',
    'Count of Safety Measures No Different',
    'Count of Safety Measures Worse',
    'Count of READM Measures Better',
    'Count of READM Measures No Different',
    'Count of READM Measures Worse',
]

df.drop(columns=cols_to_drop, inplace=True)

#clean up naming conventions to Title_Case
rename_map = {
    'MORT_participation':   'MORT_Participation',
    'Safety_participation': 'Safety_Participation',
    'READM_participation':  'READM_Participation',
    'PtExp_participation':  'PtExp_Participation',
    'TE_participation':     'TE_Participation',
    'MORT_better':          'MORT_Better',
    'Safety_better':        'Safety_Better',
    'READM_better':         'READM_Better',
}

df.rename(columns=rename_map, inplace=True)

print("Final shape:", df.shape)
print("\nFinal columns:")
for col in df.columns:
    print(f"  {col}")

Final shape: (5426, 23)

Final columns:
  Facility ID
  State
  Hospital_Type
  Birthing_Center
  Emergency_Services
  Hospital_Ownership
  Hospital_Rating
  MORT_Participation
  Safety_Participation
  READM_Participation
  PtExp_Participation
  TE_Participation
  Engagement_Index
  Domains_Reporting
  MORT_Better
  MORT_Worse
  Safety_Better
  Safety_Worse
  READM_Better
  READM_Worse
  Net_Performance_Score
  Net_Performance_Rate
  Performance_Ratio


In [22]:
#export dataset
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define a path inside your Drive
# (Everything in Drive starts with /content/drive/MyDrive/)
drive_path = '/content/drive/MyDrive/ADS 599 Capstone Project/Data/'

# 3. Ensure the folder exists
import os
if not os.path.exists(drive_path):
    os.makedirs(drive_path)

# 4. Save the file directly to your Drive
df.to_csv(os.path.join(drive_path, 'Hospital_General_Clean_Final.csv'), index=False)

print(f"File saved to your Drive at: {drive_path}")

Mounted at /content/drive
File saved to your Drive at: /content/drive/MyDrive/ADS 599 Capstone Project/Data/
